In [2]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (10).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,4,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,5,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,6,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,97,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,98,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,99,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,100,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [ ]:
df["Оценка"].value_counts()

,count
Оценка,
4,208
3,206
1,110
2,105
5,94
7,26
6,21
8,21
9,18


In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.0 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
import pickle

In [5]:
y_audience = df['ЦА'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text, y_audience, test_size=0.2, random_state=42
)
tfidf_audience = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_audience = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_audience = tfidf_audience.fit_transform(X_train_text_audience)
X_train_svd_audience = svd_audience.fit_transform(X_train_tfidf_audience)
X_test_tfidf_audience = tfidf_audience.transform(X_test_text_audience)
X_test_svd_audience = svd_audience.transform(X_test_tfidf_audience)

In [6]:
from sklearn.model_selection import GridSearchCV

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [8]:
from catboost import CatBoostClassifier

In [9]:
model_audience = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)

In [10]:
model_audience.fit(X_train_svd_audience , y_train_audience)

0:	learn: 1.5608258	total: 91.4ms	remaining: 27.3s
50:	learn: 1.1654145	total: 1.78s	remaining: 8.7s
100:	learn: 0.9748686	total: 3.48s	remaining: 6.86s
150:	learn: 0.8311179	total: 5.14s	remaining: 5.07s
200:	learn: 0.7226897	total: 7.74s	remaining: 3.81s
250:	learn: 0.6432038	total: 13s	remaining: 2.53s
299:	learn: 0.5770820	total: 16.6s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=3, early_stopping_rounds=50, eval_metric='MultiClass', iterations=300, learning_rate=0.1, loss_function='MultiClass', random_seed=42, verbose=50)

In [ ]:
y_pred_audience = model_audience.predict(X_test_svd_audience)
print(classification_report(y_test_audience, y_pred_audience))
print("MAE=",mean_absolute_error(y_test_audience, y_pred_audience))

              precision    recall  f1-score   support

           1       0.50      0.54      0.52        13
           2       0.50      0.27      0.35        22
           3       0.43      0.44      0.43        34
           4       0.53      0.40      0.46        52
           5       0.43      0.69      0.53        32
           6       0.00      0.00      0.00         2
           7       0.00      0.00      0.00         3
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         4

    accuracy                           0.44       163
   macro avg       0.26      0.26      0.25       163
weighted avg       0.45      0.44      0.43       163

MAE= 0.8159509202453987


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
y_pred_audience_train = model_audience.predict(X_train_svd_audience)
print(classification_report(y_train_audience, y_pred_audience_train))
print("MAE=",mean_absolute_error(y_train_audience, y_pred_audience_train))

              precision    recall  f1-score   support

           1       0.95      1.00      0.98       123
           2       0.92      0.95      0.94       219
           3       0.94      0.92      0.93       276
           4       0.95      0.85      0.90       324
           5       0.89      0.98      0.93       284

    accuracy                           0.93      1226
   macro avg       0.93      0.94      0.94      1226
weighted avg       0.93      0.93      0.93      1226

MAE= 0.09624796084828711


In [12]:
y_sol= df['Проработка решения'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text, y_sol, test_size=0.2, random_state=42
)
tfidf_sol = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_sol= TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_sol = tfidf_sol.fit_transform(X_train_text_sol)
X_train_svd_sol = svd_sol.fit_transform(X_train_tfidf_sol)
X_test_tfidf_sol = tfidf_sol.transform(X_test_text_sol)
X_test_svd_sol = svd_sol.transform(X_test_tfidf_sol)

In [13]:
model_sol = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_sol.fit(X_train_svd_sol , y_train_sol)
y_pred_sol = model_sol.predict(X_test_svd_sol)
#y_pred_sol
print(classification_report(y_test_sol, y_pred_sol))
print("MAE=",mean_absolute_error(y_test_sol, y_pred_sol))

0:	learn: 1.5754534	total: 42.3ms	remaining: 12.6s
50:	learn: 1.1826481	total: 1.81s	remaining: 8.82s
100:	learn: 0.9884003	total: 3.47s	remaining: 6.84s
150:	learn: 0.8530787	total: 5.14s	remaining: 5.07s
200:	learn: 0.7529632	total: 6.93s	remaining: 3.41s
250:	learn: 0.6771195	total: 12.2s	remaining: 2.38s
299:	learn: 0.6102119	total: 15.7s	remaining: 0us
              precision    recall  f1-score   support

           1       0.52      0.65      0.58        20
           2       0.58      0.43      0.50        58
           3       0.59      0.54      0.56        85
           4       0.53      0.58      0.55        79
           5       0.53      0.61      0.57        64
           6       0.00      0.00      0.00         1

    accuracy                           0.55       307
   macro avg       0.46      0.47      0.46       307
weighted avg       0.55      0.55      0.55       307

MAE= 0.5928338762214984


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
y_pred_sol_train = model_sol.predict(X_train_svd_sol)
#y_pred_sol
print(classification_report(y_train_sol, y_pred_sol_train))
print("MAE=",mean_absolute_error(y_train_sol, y_pred_sol_train))

              precision    recall  f1-score   support

           1       0.91      1.00      0.95       121
           2       0.91      0.90      0.91       252
           3       0.91      0.88      0.90       278
           4       0.94      0.87      0.90       306
           5       0.87      0.95      0.91       269

    accuracy                           0.91      1226
   macro avg       0.91      0.92      0.91      1226
weighted avg       0.91      0.91      0.91      1226

MAE= 0.1362153344208809


In [15]:
y_finance= df['Финансовая модель и метрики'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_finance, X_test_text_finance,y_train_finance, y_test_finance = train_test_split(
    X_text, y_finance, test_size=0.2, random_state=42
)
tfidf_finance = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_finance = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_finance = tfidf_finance.fit_transform(X_train_text_finance)
X_train_svd_finance = svd_finance.fit_transform(X_train_tfidf_finance)
X_test_tfidf_finance = tfidf_finance.transform(X_test_text_finance)
X_test_svd_finance = svd_finance.transform(X_test_tfidf_finance)

In [16]:
model_finance = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_finance.fit(X_train_svd_finance , y_train_finance)
y_pred_finance = model_finance.predict(X_test_svd_finance)
print(classification_report(y_test_finance, y_pred_finance))
print("MAE=",mean_absolute_error(y_test_finance, y_pred_finance))

0:	learn: 1.5698596	total: 51.6ms	remaining: 15.4s
50:	learn: 1.1120916	total: 2.93s	remaining: 14.3s
100:	learn: 0.9162874	total: 4.62s	remaining: 9.11s
150:	learn: 0.7825604	total: 6.3s	remaining: 6.21s
200:	learn: 0.6906876	total: 7.93s	remaining: 3.91s
250:	learn: 0.6206858	total: 9.59s	remaining: 1.87s
299:	learn: 0.5621082	total: 12.3s	remaining: 0us
              precision    recall  f1-score   support

           1       0.47      0.79      0.59        19
           2       0.47      0.43      0.45        70
           3       0.54      0.55      0.54        84
           4       0.73      0.58      0.65        88
           5       0.55      0.67      0.61        46

    accuracy                           0.56       307
   macro avg       0.55      0.60      0.57       307
weighted avg       0.58      0.56      0.56       307

MAE= 0.5472312703583062


In [17]:
y_pred_finance_train = model_finance.predict(X_train_svd_finance)
print(classification_report(y_train_finance, y_pred_finance_train))
print("MAE=",mean_absolute_error(y_train_finance, y_pred_finance_train))

              precision    recall  f1-score   support

           1       0.92      0.98      0.95       143
           2       0.94      0.90      0.92       275
           3       0.93      0.91      0.92       297
           4       0.95      0.89      0.92       312
           5       0.86      0.98      0.92       199

    accuracy                           0.92      1226
   macro avg       0.92      0.93      0.92      1226
weighted avg       0.92      0.92      0.92      1226

MAE= 0.11256117455138662


In [18]:
y_risks= df['Анализ рисков'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text, y_risks, test_size=0.2, random_state=42
)
tfidf_risks = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_risks = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_risks = tfidf_risks.fit_transform(X_train_text_risks)
X_train_svd_risks = svd_risks.fit_transform(X_train_tfidf_risks)
X_test_tfidf_risks = tfidf_risks.transform(X_test_text_risks)
X_test_svd_risks = svd_risks.transform(X_test_tfidf_risks)

In [19]:
model_risks = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_risks.fit(X_train_svd_risks , y_train_risks)
y_pred_risks = model_risks.predict(X_test_svd_risks)
print(classification_report(y_test_risks, y_pred_risks))
print("MAE=",mean_absolute_error(y_test_risks, y_pred_risks))

0:	learn: 1.5743669	total: 44.4ms	remaining: 13.3s
50:	learn: 1.1653024	total: 1.8s	remaining: 8.77s
100:	learn: 0.9640730	total: 3.49s	remaining: 6.87s
150:	learn: 0.8096973	total: 5.16s	remaining: 5.09s
200:	learn: 0.7131385	total: 6.86s	remaining: 3.38s
250:	learn: 0.6340185	total: 10.2s	remaining: 2s
299:	learn: 0.5737830	total: 14.1s	remaining: 0us
              precision    recall  f1-score   support

           1       0.69      0.80      0.74        30
           2       0.57      0.42      0.49        66
           3       0.53      0.55      0.54        82
           4       0.61      0.54      0.57        85
           5       0.52      0.73      0.60        44

    accuracy                           0.57       307
   macro avg       0.58      0.61      0.59       307
weighted avg       0.57      0.57      0.57       307

MAE= 0.5602605863192183


In [20]:
y_pred_risks_train = model_risks.predict(X_train_svd_risks)
print(classification_report(y_train_risks, y_pred_risks_train))
print("MAE=",mean_absolute_error(y_train_risks, y_pred_risks_train))

              precision    recall  f1-score   support

           1       0.95      0.98      0.96       177
           2       0.91      0.90      0.90       253
           3       0.94      0.88      0.91       317
           4       0.90      0.90      0.90       286
           5       0.90      0.99      0.94       193

    accuracy                           0.92      1226
   macro avg       0.92      0.93      0.92      1226
weighted avg       0.92      0.92      0.92      1226

MAE= 0.11092985318107668


In [21]:
y_proves= df['Доказательства'].values
X_text = df['Решение кейса'].fillna('').values
X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text, y_proves, test_size=0.2, random_state=42
)
tfidf_proves = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2)
svd_proves = TruncatedSVD(n_components=200, random_state=42)
X_train_tfidf_proves = tfidf_proves.fit_transform(X_train_text_proves)
X_train_svd_proves = svd_proves.fit_transform(X_train_tfidf_proves)
X_test_tfidf_proves = tfidf_proves.transform(X_test_text_proves)
X_test_svd_proves = svd_proves.transform(X_test_tfidf_proves)

In [22]:
model_proves = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=3,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    early_stopping_rounds=50,
    verbose=50,
    random_seed=42
)
model_proves.fit(X_train_svd_proves , y_train_proves)
y_pred_proves = model_proves.predict(X_test_svd_proves)
print(classification_report(y_test_proves, y_pred_proves))
print("MAE=",mean_absolute_error(y_test_proves, y_pred_proves))

0:	learn: 1.5582216	total: 45.2ms	remaining: 13.5s
50:	learn: 1.1685836	total: 1.84s	remaining: 9.01s
100:	learn: 0.9804784	total: 4.83s	remaining: 9.52s
150:	learn: 0.8452663	total: 6.78s	remaining: 6.69s
200:	learn: 0.7432807	total: 9.28s	remaining: 4.57s
250:	learn: 0.6599580	total: 12.7s	remaining: 2.48s
299:	learn: 0.6026153	total: 15.1s	remaining: 0us
              precision    recall  f1-score   support

           1       0.77      0.74      0.75        31
           2       0.63      0.58      0.60        71
           3       0.47      0.48      0.48        60
           4       0.62      0.58      0.60        91
           5       0.50      0.60      0.55        53
           6       0.00      0.00      0.00         1

    accuracy                           0.58       307
   macro avg       0.50      0.50      0.50       307
weighted avg       0.58      0.58      0.58       307

MAE= 0.5928338762214984


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
y_pred_proves_train = model_proves.predict(X_train_svd_proves)
print(classification_report(y_train_proves, y_pred_proves_train))
print("MAE=",mean_absolute_error(y_train_proves, y_pred_proves_train))

              precision    recall  f1-score   support

           1       0.94      0.98      0.96       175
           2       0.94      0.93      0.93       252
           3       0.93      0.88      0.90       260
           4       0.94      0.90      0.92       303
           5       0.89      0.97      0.93       236

    accuracy                           0.93      1226
   macro avg       0.93      0.93      0.93      1226
weighted avg       0.93      0.93      0.93      1226

MAE= 0.10522022838499184


In [24]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [25]:
def get_prediction_audience(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_audience.transform([text])
  text_svd = svd_audience.transform(text_tfidf)
  score = model_audience.predict(text_svd)[0]
  return int(score)

In [26]:
def get_prediction_solution(text):
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_sol.transform([text])
  text_svd = svd_sol.transform(text_tfidf)
  score = model_sol.predict(text_svd)[0]
  return int(score)

In [27]:
def get_prediction_finance(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_finance.transform([text])
  text_svd = svd_finance.transform(text_tfidf)
  score = model_finance.predict(text_svd)[0]
  return int(score)

In [28]:
def get_prediction_risks(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_risks.transform([text])
  text_svd = svd_risks.transform(text_tfidf)
  score = model_risks.predict(text_svd)[0]
  return int(score)

In [29]:
def get_prediction_proves(text):

  if pd.isna(text) or not isinstance(text, str):
        text = ""
  text_tfidf = tfidf_proves.transform([text])
  text_svd = svd_proves.transform(text_tfidf)
  score = model_proves.predict(text_svd)[0]
  return int(score)

In [30]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

/tmp/ipykernel_948/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_948/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_948/131350308.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_948/2249487333.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a sin

In [31]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [32]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.5538461538461539
f1micro_audience=  0.5538461538461539
f1macro_audience=  0.542480680971247
              precision    recall  f1-score   support

           1       0.57      0.74      0.64        23
           2       0.73      0.31      0.43        26
           3       0.58      0.62      0.60        34
           4       0.47      0.64      0.54        22
           5       0.52      0.48      0.50        25

    accuracy                           0.55       130
   macro avg       0.57      0.56      0.54       130
weighted avg       0.58      0.55      0.54       130

MAE= 0.6230769230769231


In [33]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.36923076923076925
f1micro_sol=  0.36923076923076925
f1macro_sol=  0.32973672573672574
              precision    recall  f1-score   support

           1       0.45      0.95      0.62        21
           2       0.33      0.03      0.06        32
           3       0.40      0.27      0.32        30
           4       0.29      0.56      0.38        25
           5       0.36      0.23      0.28        22

    accuracy                           0.37       130
   macro avg       0.37      0.41      0.33       130
weighted avg       0.36      0.37      0.31       130

MAE= 0.7923076923076923


In [34]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.46153846153846156
f1micro_finance=  0.46153846153846156
f1macro_finance=  0.45149756133362684
              precision    recall  f1-score   support

           1       0.53      0.87      0.66        23
           2       0.33      0.21      0.26        33
           3       0.44      0.48      0.46        31
           4       0.41      0.54      0.46        24
           5       1.00      0.26      0.42        19

    accuracy                           0.46       130
   macro avg       0.54      0.47      0.45       130
weighted avg       0.50      0.46      0.44       130

MAE= 0.6307692307692307


In [35]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.4307692307692308
f1micro_risks=  0.4307692307692308
f1macro_risks=  0.3200131187480966
              precision    recall  f1-score   support

           1       0.52      0.85      0.65        26
           2       0.00      0.00      0.00        32
           3       0.37      0.63      0.47        30
           4       0.43      0.56      0.48        27
           5       0.00      0.00      0.00        15

    accuracy                           0.43       130
   macro avg       0.26      0.41      0.32       130
weighted avg       0.28      0.43      0.34       130

MAE= 0.7076923076923077


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.4153846153846154
f1micro_proves=  0.4153846153846154
f1macro_proves=  0.3857435570872041
              precision    recall  f1-score   support

           1       0.43      0.72      0.54        18
           2       0.55      0.47      0.51        38
           3       0.40      0.08      0.13        25
           4       0.29      0.68      0.41        19
           5       0.47      0.27      0.34        30

    accuracy                           0.42       130
   macro avg       0.43      0.45      0.39       130
weighted avg       0.45      0.42      0.39       130

MAE 0.7769230769230769


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [37]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [38]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.5153846153846153


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
